# Stage 3 — Generalization test (M vs B4), pure inference, no training

**2026-09-19/30.** Ayesha ka sawal: M (residual PAH) training ke dauran
zyada deterministic dikha (lower entropy) B4 (fixed alpha=0.5) se — kya ye
matlab rakhta hai? Aur, in already-trained Stage 3 seed-42 models ko sirf
**generalization** ke liye test kar sakte hain kya?

**Ye notebook koi naya training nahi karta.** Dono models (`var-alpha` M
aur `fixed-alpha` B4, Stage 3 seed-42 se, already trained) load karta hai
aur **sirf inference** (koi gradient update nahi) se do cheezein test
karta hai:

1. **(A) In-distribution generalization** — 5 bilkul naye eval seeds
   (`2001-2005`, kabhi training ya training-time-eval mein use nahi hue —
   training seeds the `42`, training-time eval seed tha `999`), **same**
   config jis pe train hua (8 drones, 10 obstacles).
2. **(B) Out-of-distribution generalization** — wahi naye seeds, lekin
   **harder** config (8 drones, **16 obstacles** — 60% zyada, kabhi nahi
   dekha training mein). Ye seedha Ayesha ke hypothesis ko test karta hai:
   kya M (kam entropy, kam explore kiya) B4 se zyada degrade karta hai jab
   conditions genuinely badalte hain?

**Methodology note (important, honestly flag karna hai):** ye eval
**deterministic** hai (action = policy ka mean, koi random sampling nahi)
— jaisa `docs/plans/02_experiment_protocol.md` mein likha hai sahi
tareeka. Lekin poore session bhar (Stage 2 aur Stage 3 training runs ke
dauran) jo bhi success/collision numbers report hue, wo **stochastic**
(`dist.sample()`) actions se the, deterministic nahi — `agent.get_actions()`
andar hamesha sample karta hai. Matlab is notebook ke numbers **training
runs ke live-log numbers se directly comparable nahi hain** (alag
evaluation tareeka) — ye apne andar consistent hai (M aur B4 dono isi
tareeke se yahan eval hue), bas purane numbers se seedha match nahi karega.

**Results already ek baar local pe chal chuke hain** (2026-09-19,
throwaway venv) — ye notebook usi test ko permanent, reproducible form mein
save karta hai. Chalane ke liye koi Kaggle GPU nahi chahiye — pure
inference hai, CPU pe bhi chhote se minutes mein complete hota hai.


In [ ]:
# Cell 1 — Imports
import torch
import numpy as np
from scipy.optimize import linear_sum_assignment
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {DEVICE}')

In [ ]:
# Cell 2 — Conflict Graph
# Ported from code/algorithms/conflict_graph.py

import numpy as np

def _time_to_cpa(pos_i, pos_j, vel_i, vel_j):
    p = pos_i - pos_j
    v = vel_i - vel_j
    vv = np.dot(v, v)
    if vv < 1e-8:
        return np.inf
    return -np.dot(p, v) / vv

def _dist_at_cpa(pos_i, pos_j, vel_i, vel_j, horizon):
    p = pos_i - pos_j
    v = vel_i - vel_j
    vv = np.dot(v, v)
    if vv < 1e-8:
        return float(np.linalg.norm(p)), 0.0
    t_star = -np.dot(p, v) / vv
    t_clamped = float(np.clip(t_star, 0.0, horizon))
    return float(np.linalg.norm(p + v * t_clamped)), t_star

class ConflictGraph:
    K_NBR = 4

    def __init__(self, n_drones, horizon=3.0, d_danger=9.0):
        self.n      = n_drones
        self.H      = horizon
        self.d_dan  = d_danger
        self.adj    = np.zeros((n_drones, n_drones), dtype=bool)
        self.t_mat  = np.full((n_drones, n_drones), np.inf)

    def update(self, pos, vel):
        self.adj[:] = False
        self.t_mat[:] = np.inf
        for i in range(self.n):
            for j in range(i + 1, self.n):
                dcpa, t_star = _dist_at_cpa(pos[i], pos[j], vel[i], vel[j], self.H)
                self.t_mat[i, j] = self.t_mat[j, i] = t_star
                if dcpa < self.d_dan and 0.0 <= t_star <= self.H:
                    self.adj[i, j] = self.adj[j, i] = True

    def n_conflict(self, i):
        return int(self.adj[i].sum())

    def tau_collision(self, i):
        nbrs = np.where(self.adj[i])[0]
        if len(nbrs) == 0:
            return self.H
        return float(np.min(self.t_mat[i, nbrs]))

    def neighbor_obs(self, i, pos, vel):
        obs = np.zeros(self.K_NBR * 5, dtype=np.float32)
        nbrs = np.where(self.adj[i])[0]
        if len(nbrs) == 0:
            return obs
        order = np.argsort(self.t_mat[i, nbrs])
        nbrs  = nbrs[order]
        for slot, j in enumerate(nbrs[:self.K_NBR]):
            b = slot * 5
            obs[b:b+2]   = pos[j] - pos[i]
            obs[b+2:b+4] = vel[j] - vel[i]
            obs[b+4]     = 1.0
        return obs

In [ ]:
# Cell 3 — Environment (same as Stage 2b, + pah_inputs in step() info dict)
#
# Only addition vs Stage 2b: step() now also returns info['pah_inputs'] =
# {tau, d_target, n_conflict} per drone, computed from the conflict graph
# (already updated every step) and the current Hungarian assignment. This is
# exactly what code/environment/multi_uav_env.py already does — PAH needs
# these three numbers per drone, per step.

import numpy as np
from scipy.optimize import linear_sum_assignment

K_NBR    = 4
BASE_DIM = 10
OBS_DIM  = BASE_DIM + K_NBR * 5  # 30

class MultiUAVEnv:
    def __init__(self, n_drones=5, n_obstacles=5, world_size=500.0,
                 max_speed=5.0, max_steps=500, collision_radius=3.0,
                 target_radius=25.0, success_bonus=20.0,
                 target_speed=1.0, seed=None):
        self.n         = n_drones
        self.n_obs     = n_obstacles
        self.W         = world_size
        self.max_sp    = max_speed
        self.max_steps = max_steps
        self.col_r     = collision_radius
        self.tgt_r     = target_radius
        self.bonus     = success_bonus
        self.tgt_speed = target_speed
        self.rng       = np.random.default_rng(seed)
        self.cg        = ConflictGraph(n_drones, horizon=3.0,
                                       d_danger=collision_radius * 3.0)
        self.obs_dim   = OBS_DIM
        self.act_dim   = 2

    def reset(self):
        self.t                  = 0
        self.collision_occurred = False
        all_pos = self._sample_non_overlapping(self.n * 2, self.col_r * 2)
        self.drone_pos  = all_pos[:self.n].copy()
        self.target_pos = all_pos[self.n:].copy()
        self.obstacle_pos = self._place_obstacles(
            np.concatenate([self.drone_pos, self.target_pos]), self.n_obs
        )
        self.drone_vel = np.zeros((self.n, 2), dtype=np.float32)
        angles = self.rng.uniform(0, 2 * np.pi, self.n)
        self.target_vel = (self.tgt_speed * np.stack(
            [np.cos(angles), np.sin(angles)], axis=1
        )).astype(np.float32)
        self.assignment = self._hungarian()
        self.cg.update(self.drone_pos, self.drone_vel)
        return self._obs()

    def step(self, actions):
        self.t += 1
        actions = np.clip(actions, -self.max_sp, self.max_sp)
        self.drone_vel = actions.astype(np.float32)
        self.drone_pos = np.clip(self.drone_pos + self.drone_vel, 0.0, self.W)

        self.target_pos = self.target_pos + self.target_vel
        for k in range(self.n):
            for dim in range(2):
                if self.target_pos[k, dim] < 10.0:
                    self.target_pos[k, dim] = 10.0
                    self.target_vel[k, dim] *= -1.0
                elif self.target_pos[k, dim] > self.W - 10.0:
                    self.target_pos[k, dim] = self.W - 10.0
                    self.target_vel[k, dim] *= -1.0

        self.assignment = self._hungarian()
        self.cg.update(self.drone_pos, self.drone_vel)

        rewards, r_mission, r_safety = self._rewards()
        all_reached   = self._all_reached()
        any_collision = self._any_collision()
        timeout       = self.t >= self.max_steps

        if any_collision:
            self.collision_occurred = True

        terminated = all_reached or any_collision
        truncated  = timeout

        if all_reached and self.bonus:
            rewards   = rewards + self.bonus
            r_mission = r_mission + self.bonus   # bonus is a mission outcome, not safety

        # PAH inputs — per drone, from the conflict graph + current assignment.
        pah_inputs = {
            'tau': np.array([self.cg.tau_collision(i) for i in range(self.n)], dtype=np.float32),
            'd_target': np.array([
                np.linalg.norm(self.drone_pos[i] - self.target_pos[self.assignment[i]])
                for i in range(self.n)
            ], dtype=np.float32),
            'n_conflict': np.array([self.cg.n_conflict(i) for i in range(self.n)], dtype=np.float32),
        }

        info = {
            'all_targets_reached': all_reached,
            'any_collision':       any_collision,
            'episode_collision':   self.collision_occurred,
            'r_mission':           r_mission,
            'r_safety':            r_safety,
            'pah_inputs':          pah_inputs,
        }
        return self._obs(), rewards, terminated, truncated, info

    def _obs(self):
        obs = np.zeros((self.n, self.obs_dim), dtype=np.float32)
        for i in range(self.n):
            rel = self.target_pos[self.assignment[i]] - self.drone_pos[i]
            cl  = self._clearances(i)
            base = np.concatenate([
                self.drone_pos[i] / self.W,
                self.drone_vel[i] / self.max_sp,
                rel               / self.W,
                cl,
            ])
            nbr = self.cg.neighbor_obs(i, self.drone_pos, self.drone_vel)
            for s in range(K_NBR):
                nbr[s*5 : s*5+2]   /= self.W
                nbr[s*5+2 : s*5+4] /= self.max_sp
            obs[i] = np.concatenate([base, nbr])
        return obs

    def _rewards(self):
        r_mission = np.zeros(self.n, dtype=np.float32)
        r_safety  = np.zeros(self.n, dtype=np.float32)
        d_danger  = self.col_r * 3.0
        zone_w    = d_danger - self.col_r
        for i in range(self.n):
            rel  = self.target_pos[self.assignment[i]] - self.drone_pos[i]
            dist = np.linalg.norm(rel)
            direction  = rel / (dist + 1e-6)
            r_progress = float(np.dot(self.drone_vel[i], direction) / self.max_sp)
            r_mission[i] = 0.4 * r_progress + 0.3 * (-dist / self.W)
            min_d = float('inf')
            for j in range(self.n):
                if j != i:
                    min_d = min(min_d, np.linalg.norm(self.drone_pos[i] - self.drone_pos[j]))
            for op in self.obstacle_pos:
                min_d = min(min_d, np.linalg.norm(self.drone_pos[i] - op))
            if min_d < self.col_r:
                r_safety[i] = -1.0
            elif min_d < d_danger:
                r_safety[i] = -((d_danger - min_d) / zone_w)
        rewards = r_mission + 0.3 * r_safety
        return rewards, r_mission, r_safety

    def _hungarian(self):
        cost = np.array([
            [np.linalg.norm(self.drone_pos[i] - self.target_pos[j])
             for j in range(self.n)]
            for i in range(self.n)
        ])
        _, col = linear_sum_assignment(cost)
        return col

    def _all_reached(self):
        return all(
            np.linalg.norm(self.drone_pos[i] - self.target_pos[self.assignment[i]]) <= self.tgt_r
            for i in range(self.n)
        )

    def _any_collision(self):
        for i in range(self.n):
            for j in range(self.n):
                if j != i and np.linalg.norm(self.drone_pos[i] - self.drone_pos[j]) < self.col_r:
                    return True
            for op in self.obstacle_pos:
                if np.linalg.norm(self.drone_pos[i] - op) < self.col_r:
                    return True
        return False

    def _clearances(self, i):
        pos = self.drone_pos[i]
        cl  = np.array([self.W - pos[1], pos[1], self.W - pos[0], pos[0]], dtype=np.float32)
        for op in self.obstacle_pos:
            diff = op - pos
            if diff[1] > 0: cl[0] = min(cl[0],  diff[1])
            else:           cl[1] = min(cl[1], -diff[1])
            if diff[0] > 0: cl[2] = min(cl[2],  diff[0])
            else:           cl[3] = min(cl[3], -diff[0])
        return cl / self.W

    def _sample_non_overlapping(self, n, min_dist):
        pts, attempts = [], 0
        while len(pts) < n:
            attempts += 1
            if attempts > 10000:
                raise RuntimeError('World too crowded.')
            c = self.rng.uniform(10.0, self.W - 10.0, 2)
            if not any(np.linalg.norm(c - p) < min_dist for p in pts):
                pts.append(c)
        return np.array(pts, dtype=np.float32)

    def _place_obstacles(self, existing, n_obs, max_tries=1000):
        if n_obs <= 0:
            return np.zeros((0, 2), dtype=np.float32)
        r_min = 2.0 * self.col_r
        placed = []
        for _ in range(n_obs):
            for _ in range(max_tries):
                c = self.rng.uniform(10.0, self.W - 10.0, 2)
                if any(np.linalg.norm(c - o) < r_min for o in placed): continue
                if any(np.linalg.norm(c - p) < self.col_r for p in existing): continue
                placed.append(c)
                break
            else:
                break
        return np.array(placed, dtype=np.float32) if placed else np.zeros((0, 2), dtype=np.float32)

print('obs_dim = {}  (10 base + {}x5 conflict neighbors, normalized)'.format(OBS_DIM, K_NBR))
print('step() now also returns info["pah_inputs"] = {tau, d_target, n_conflict}')


In [ ]:
# Cell 4 -- MAPPO + PAH (prior + learned residual, single critic) -- Stage 3, 8 drones
#
# ADOPTED design (Ayesha + Manish sign-off, sessions/2026-09-18.md Section
# 22) -- see Cell 0 markdown for Stage 3 context (freeze-first critic
# warm-start, disjoint eval seed). Single-critic architecture, same
# Actor/Critic/RolloutBuffer as Stage 2 (unchanged, proven). PAH is the
# prior+residual design:
#
#   alpha(s) = clip( alpha_safe(tau) + delta_phi(s), alpha_min, alpha_max )
#
# alpha_safe(tau) is the SAME tau-formula stage2-pah-v0/v2 tried to pull
# alpha toward with a prior LOSS -- now it's hardcoded directly into
# forward(), not something the network has to learn or that a loss has to
# win a race for. delta_phi(s) is a small learned correction, bounded via
# tanh to [-delta_scale, +delta_scale], regularized toward zero.
#
# Why this fixes the seed-dependent race/lock-in diagnosed in
# sessions/2026-09-18.md Section 15: previously the WHOLE of alpha came
# from the network, and both the actor loss (pulls alpha up, batch-
# averaged, dominated by common safe states) and the prior loss (pulls
# alpha down, only near danger) fought over that one number before the
# policy's entropy collapsed -- whichever won got locked in, and which one
# won was sensitive to each seed's early rollout trajectory. Now alpha_safe
# is structurally always present; the actor loss can only ever push the
# bounded residual around it, not erase it.

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.mu_head = nn.Linear(hidden, act_dim)
        self.log_std = nn.Parameter(torch.zeros(act_dim))

    def forward(self, x):
        h       = self.net(x)
        mu      = self.mu_head(h)
        log_std = self.log_std.clamp(-4.0, 1.5)   # safety net, see Stage 2 notes
        std     = log_std.exp().expand_as(mu)
        return Normal(mu, std)

    def get_action(self, obs):
        dist   = self(obs)
        action = dist.sample()
        log_p  = dist.log_prob(action).sum(-1)
        return action, log_p

    def evaluate_action(self, obs, action):
        dist    = self(obs)
        log_p   = dist.log_prob(action).sum(-1)
        entropy = dist.entropy().sum(-1)
        return log_p, entropy


class Critic(nn.Module):
    """Centralized critic -- one value per timestep from the flattened joint state."""
    def __init__(self, global_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(global_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),     nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, global_state):
        return self.net(global_state).squeeze(-1)


class PAHNormalizer:
    """Scales the 3 raw PAH inputs to roughly [0,1]. Ported from code/algorithms/pah.py."""
    def __init__(self, horizon=3.0, d_max=707.1, n_drones=5):
        self.horizon  = horizon
        self.d_max    = d_max
        self.n_drones = n_drones

    def normalize(self, tau, d_target, n_conflict):
        tau_norm = tau.clamp(0.0, self.horizon) / self.horizon
        d_norm   = d_target.clamp(0.0, self.d_max) / self.d_max
        denom    = max(self.n_drones - 1, 1)
        n_norm   = n_conflict.clamp(0.0, denom) / denom
        return torch.stack([tau_norm, d_norm, n_norm], dim=-1)


class PriorityArbitrationHead(nn.Module):
    """alpha(s) = clip(alpha_safe(tau) + delta(s), alpha_min, alpha_max).
    alpha_safe is a hardcoded tau-formula (never learned, never lost).
    delta is a small learned correction, tanh-bounded to
    [-delta_scale, +delta_scale], regularized toward zero. See Cell 4
    header for the full rationale."""
    def __init__(self, hidden_dim=32, alpha_min=0.1, alpha_max=0.9,
                 delta_scale=0.2, prior_coef=0.15):
        super().__init__()
        self.alpha_min   = alpha_min
        self.alpha_max   = alpha_max
        self.delta_scale = delta_scale
        self.prior_coef  = prior_coef
        self.net = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        nn.init.zeros_(self.net[-1].bias)
        nn.init.xavier_uniform_(self.net[-1].weight, gain=0.1)

    def alpha_safe(self, tau_norm):
        return self.alpha_min + (self.alpha_max - self.alpha_min) * tau_norm

    def forward(self, x):
        """x: (..., 3) = [tau_norm, d_norm, n_norm]. Returns (alpha, delta)."""
        tau_norm  = x[..., 0]
        raw_delta = torch.tanh(self.net(x).squeeze(-1))
        delta     = self.delta_scale * raw_delta
        alpha     = (self.alpha_safe(tau_norm) + delta).clamp(self.alpha_min, self.alpha_max)
        return alpha, delta

    def compute_prior_loss(self, delta):
        """Regularizes the residual toward zero -- NOT toward the formula
        (alpha_safe already IS the formula, hardcoded). This only earns a
        role in training if a non-zero delta demonstrably lowers the total
        loss beyond what alpha_safe already gets for free."""
        if self.prior_coef == 0.0:
            return torch.tensor(0.0, device=delta.device)
        return self.prior_coef * delta.pow(2).mean()


class RolloutBuffer:
    """Per-drone rewards/log-probs/PAH-inputs kept separate throughout -- nothing
    here is averaged across drones (see sessions/2026-09-15.md Part 12 for why
    that was a bug in an earlier version of the Stage 2 notebook)."""
    def __init__(self, n_drones):
        self.n_drones = n_drones
        self.clear()

    def clear(self):
        self.obs, self.acts, self.rews = [], [], []
        self.vals, self.lps, self.dones = [], [], []
        self.r_mission, self.r_safety = [], []
        self.tau, self.d_target, self.n_conflict = [], [], []

    def add(self, obs, act, rew, val, lp, done, r_mission, r_safety, pah_inputs):
        self.obs.append(obs.copy())
        self.acts.append(act.copy())
        self.rews.append(rew.copy())
        self.vals.append(float(val))
        self.lps.append(lp.copy())
        self.dones.append(bool(done))
        self.r_mission.append(r_mission.copy())
        self.r_safety.append(r_safety.copy())
        self.tau.append(pah_inputs['tau'].copy())
        self.d_target.append(pah_inputs['d_target'].copy())
        self.n_conflict.append(pah_inputs['n_conflict'].copy())

    def __len__(self):
        return len(self.rews)

    def compute_gae(self, last_val, gamma, lam):
        """Combined-reward GAE -- used for the critic's return target."""
        T = len(self.rews)
        n = self.n_drones
        rewards = np.array(self.rews,  dtype=np.float32)
        values  = np.array(self.vals,  dtype=np.float32)
        dones   = np.array(self.dones, dtype=np.float32)
        adv      = np.zeros((T, n), dtype=np.float32)
        last_gae = np.zeros(n, dtype=np.float32)
        for t in reversed(range(T)):
            next_val = last_val if t == T - 1 else values[t + 1]
            mask     = 1.0 - dones[t]
            delta    = rewards[t] + gamma * next_val * mask - values[t]
            last_gae = delta + gamma * lam * mask * last_gae
            adv[t]   = last_gae
        returns = adv + values[:, np.newaxis]
        return adv, returns

    def compute_gae_components(self, last_val, gamma, lam):
        """Separate GAE for r_mission and r_safety, sharing the combined-reward
        critic as baseline ("Option B lite" -- see docs/research/01_pah_design.md
        Section 9). Used for the alpha-weighted actor loss."""
        T = len(self.rews)
        n = self.n_drones
        rm      = np.array(self.r_mission, dtype=np.float32)
        rs      = np.array(self.r_safety,  dtype=np.float32)
        values  = np.array(self.vals,      dtype=np.float32)
        dones   = np.array(self.dones,     dtype=np.float32)
        adv_m = np.zeros((T, n), dtype=np.float32)
        adv_s = np.zeros((T, n), dtype=np.float32)
        gae_m = np.zeros(n, dtype=np.float32)
        gae_s = np.zeros(n, dtype=np.float32)
        for t in reversed(range(T)):
            next_val = last_val if t == T - 1 else values[t + 1]
            mask     = 1.0 - dones[t]
            delta_m  = rm[t] + gamma * next_val * mask - values[t]
            delta_s  = rs[t] + gamma * next_val * mask - values[t]
            gae_m    = delta_m + gamma * lam * mask * gae_m
            gae_s    = delta_s + gamma * lam * mask * gae_s
            adv_m[t] = gae_m
            adv_s[t] = gae_s
        return adv_m, adv_s


class MAPPOAgent:
    def __init__(self, n_drones, obs_dim, act_dim, use_pah=True, fixed_alpha=0.5,
                 pah_hidden=32, alpha_min=0.1, alpha_max=0.9, pah_delta_scale=0.2,
                 pah_prior_coef=0.15, pah_horizon=3.0, pah_d_max=707.1,
                 lr=3e-4, gamma=0.99, lam=0.95, clip_eps=0.2, vf_coef=0.5,
                 ent_coef=0.003, n_epochs=10, batch_size=64):
        self.n_drones = n_drones
        self.use_pah  = use_pah
        self.fixed_alpha = fixed_alpha
        self.gamma, self.lam           = gamma, lam
        self.clip_eps                  = clip_eps
        self.vf_coef, self.ent_coef    = vf_coef, ent_coef
        self.n_epochs, self.batch_size = n_epochs, batch_size

        global_dim  = n_drones * obs_dim
        self.actor  = Actor(obs_dim, act_dim).to(DEVICE)
        self.critic = Critic(global_dim).to(DEVICE)
        params = list(self.actor.parameters()) + list(self.critic.parameters())

        if use_pah:
            self.pah        = PriorityArbitrationHead(
                pah_hidden, alpha_min, alpha_max, pah_delta_scale, pah_prior_coef
            ).to(DEVICE)
            self.normalizer = PAHNormalizer(pah_horizon, pah_d_max, n_drones)
            params += list(self.pah.parameters())
        else:
            self.pah        = None
            self.normalizer = None

        self.opt = optim.Adam(params, lr=lr)

    @torch.no_grad()
    def get_actions(self, obs):
        obs_t          = torch.tensor(obs, dtype=torch.float32, device=DEVICE)
        action_t, lp_t = self.actor.get_action(obs_t)
        global_state   = obs_t.flatten().unsqueeze(0)
        value          = self.critic(global_state).item()
        return action_t.cpu().numpy(), lp_t.cpu().numpy(), value

    @torch.no_grad()
    def compute_alpha(self, tau_np, d_np, n_np):
        """Inference-time alpha (no gradient) -- used to build the combined
        reward that the environment/buffer stores for the critic target."""
        if not self.use_pah:
            return np.full(self.n_drones, self.fixed_alpha, dtype=np.float32)
        tau = torch.tensor(tau_np, dtype=torch.float32, device=DEVICE)
        d   = torch.tensor(d_np,   dtype=torch.float32, device=DEVICE)
        n   = torch.tensor(n_np,   dtype=torch.float32, device=DEVICE)
        x         = self.normalizer.normalize(tau, d, n)
        alpha, _  = self.pah(x)
        return alpha.cpu().numpy()

    def _alpha_with_grad(self, tau_t, d_t, n_t):
        """Training-time (alpha, delta) with gradient. delta is what
        compute_prior_loss regularizes toward zero -- NOT alpha itself, and
        NOT toward any formula (alpha_safe inside pah.forward already IS
        the formula)."""
        if not self.use_pah:
            alpha = torch.full((tau_t.shape[0],), self.fixed_alpha,
                                dtype=torch.float32, device=DEVICE)
            return alpha, None
        x = self.normalizer.normalize(tau_t, d_t, n_t)
        return self.pah(x)   # (alpha, delta)

    def update(self, buffer, last_obs):
        last_obs_t = torch.tensor(last_obs, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            last_val = self.critic(last_obs_t.flatten().unsqueeze(0)).item()

        adv_np, ret_np       = buffer.compute_gae(last_val, self.gamma, self.lam)
        adv_m_np, adv_s_np   = buffer.compute_gae_components(last_val, self.gamma, self.lam)

        obs_np = np.array(buffer.obs,  dtype=np.float32)   # (T,n,D)
        act_np = np.array(buffer.acts, dtype=np.float32)   # (T,n,2)
        lp_np  = np.array(buffer.lps,  dtype=np.float32)   # (T,n)
        tau_np = np.array(buffer.tau,        dtype=np.float32)  # (T,n)
        d_np   = np.array(buffer.d_target,   dtype=np.float32)  # (T,n)
        nc_np  = np.array(buffer.n_conflict, dtype=np.float32)  # (T,n)
        T, N, D = obs_np.shape

        adv_t = torch.tensor(adv_np, dtype=torch.float32, device=DEVICE)
        ret_t = torch.tensor(ret_np, dtype=torch.float32, device=DEVICE)
        adv_flat_all = adv_t.reshape(-1)
        adv_t = (adv_t - adv_flat_all.mean()) / (adv_flat_all.std() + 1e-8)  # unused directly but kept for parity/logging

        adv_m_t = torch.tensor(adv_m_np, dtype=torch.float32, device=DEVICE).reshape(-1)
        adv_s_t = torch.tensor(adv_s_np, dtype=torch.float32, device=DEVICE).reshape(-1)
        adv_m_flat = (adv_m_t - adv_m_t.mean()) / (adv_m_t.std() + 1e-8)
        adv_s_flat = (adv_s_t - adv_s_t.mean()) / (adv_s_t.std() + 1e-8)

        obs_flat = torch.tensor(obs_np.reshape(T * N, D), dtype=torch.float32, device=DEVICE)
        act_flat = torch.tensor(act_np.reshape(T * N, -1), dtype=torch.float32, device=DEVICE)
        lp_flat  = torch.tensor(lp_np.reshape(T * N),      dtype=torch.float32, device=DEVICE)
        ret_flat = ret_t.reshape(T * N)
        tau_flat = torch.tensor(tau_np.reshape(T * N), dtype=torch.float32, device=DEVICE)
        d_flat   = torch.tensor(d_np.reshape(T * N),   dtype=torch.float32, device=DEVICE)
        nc_flat  = torch.tensor(nc_np.reshape(T * N),  dtype=torch.float32, device=DEVICE)

        gs_t   = obs_flat.reshape(T, N * D)
        gs_exp = gs_t.unsqueeze(1).expand(T, N, -1).reshape(T * N, -1)

        idx_all = np.arange(T * N)
        a_losses, v_losses, entropies, pah_losses, alphas_seen, deltas_seen = [], [], [], [], [], []

        for _ in range(self.n_epochs):
            np.random.shuffle(idx_all)
            for start in range(0, T * N, self.batch_size):
                idx = idx_all[start:start + self.batch_size]

                log_p, entropy = self.actor.evaluate_action(obs_flat[idx], act_flat[idx])

                alpha, delta = self._alpha_with_grad(tau_flat[idx], d_flat[idx], nc_flat[idx])  # (B,), (B,) or None
                w_adv = alpha * adv_m_flat[idx] + (1.0 - alpha) * adv_s_flat[idx]

                ratio  = (log_p - lp_flat[idx]).exp()
                surr1  = ratio * w_adv
                surr2  = ratio.clamp(1 - self.clip_eps, 1 + self.clip_eps) * w_adv
                a_loss = -torch.min(surr1, surr2).mean()

                v_pred = self.critic(gs_exp[idx])
                v_loss = 0.5 * (v_pred - ret_flat[idx]).pow(2).mean()

                e_loss = -entropy.mean()
                loss = a_loss + self.vf_coef * v_loss + self.ent_coef * e_loss

                if self.use_pah:
                    pah_prior = self.pah.compute_prior_loss(delta)
                    loss = loss + pah_prior
                    pah_losses.append(pah_prior.item())
                    deltas_seen.append(delta.detach().abs().mean().item())

                self.opt.zero_grad()
                loss.backward()
                params = list(self.actor.parameters()) + list(self.critic.parameters())
                if self.use_pah:
                    params += list(self.pah.parameters())
                nn.utils.clip_grad_norm_(params, 0.5)
                self.opt.step()

                a_losses.append(a_loss.item())
                v_losses.append(v_loss.item())
                entropies.append(-e_loss.item())
                alphas_seen.append(alpha.detach().mean().item())

        stats = {
            'actor_loss':  float(np.mean(a_losses)),
            'critic_loss': float(np.mean(v_losses)),
            'entropy':     float(np.mean(entropies)),
            'alpha_mean':  float(np.mean(alphas_seen)),
            'alpha_std':   float(np.std(alphas_seen)),
        }
        if pah_losses:
            stats['pah_loss']       = float(np.mean(pah_losses))
            stats['delta_mean_abs'] = float(np.mean(deltas_seen))
        return stats

    def save(self, path):
        ckpt = {'actor': self.actor.state_dict(), 'critic': self.critic.state_dict()}
        if self.use_pah:
            ckpt['pah'] = self.pah.state_dict()
        torch.save(ckpt, path)

    def load_actor_critic(self, path):
        """Warm-start: load actor+critic (only valid when the checkpoint's
        critic input size matches THIS agent's n_drones -- e.g. same-stage
        reruns). NOT used for Stage 3 (see load_actor_only below) -- kept
        for reference/other stages."""
        ckpt = torch.load(path, map_location=DEVICE, weights_only=True)
        self.actor.load_state_dict(ckpt['actor'])
        self.critic.load_state_dict(ckpt['critic'])

    def load_actor_only(self, path):
        """Warm-start: load ONLY the actor. Used for Stage 3 -- Stage 2b's
        critic took a 5-drone joint state (5*30=150-dim); Stage 3 has 8
        drones (8*30=240-dim), so the critic CANNOT be loaded (shape
        mismatch) and starts fresh random instead. The actor transfers
        cleanly because obs_dim=30 is drone-count-agnostic (K_NBR=4 fixed
        neighbor slots) -- see Cell 0 markdown."""
        ckpt = torch.load(path, map_location=DEVICE, weights_only=True)
        self.actor.load_state_dict(ckpt['actor'])


print('MAPPO + PAH (prior + learned residual, single critic) loaded -- Stage 3, ADOPTED design.')
print('alpha(s) = clip(alpha_safe(tau) + delta(s), alpha_min, alpha_max) -- alpha_safe is '
      'hardcoded (never learned, never lost), delta is a small learned correction regularized '
      'toward zero.')


In [ ]:
# Cell 5 -- Load both seed-42 checkpoints (M and B4), pure inference (no training)

import os

N_DRONES_TRAIN = 8

# Paths relative to this notebook (code/notebooks/v3/stage3/generalization_test.ipynb)
M_PATH  = '../var-alpha/output/final_model.pt'
B4_PATH = '../fixed-alpha/output/final_model.pt'

for _p in (M_PATH, B4_PATH):
    if not os.path.exists(_p):
        raise FileNotFoundError(
            "Missing checkpoint: {}\n"
            "This notebook needs both Stage 3 seed-42 final_model.pt files "
            "(var-alpha/output/ and fixed-alpha/output/) -- run the Stage 3 "
            "training notebooks first, or place the trained checkpoints in "
            "these locations.".format(_p)
        )

def load_model(path, use_pah):
    """Loads actor (+ PAH if use_pah) from a saved MAPPOAgent checkpoint,
    for INFERENCE ONLY -- .eval() mode, no optimizer, no gradients."""
    ckpt = torch.load(path, map_location=DEVICE, weights_only=True)
    actor = Actor(OBS_DIM, 2).to(DEVICE)
    actor.load_state_dict(ckpt['actor'])
    actor.eval()
    pah, normalizer = None, None
    if use_pah:
        pah = PriorityArbitrationHead(32, 0.1, 0.9, 0.2, 0.15).to(DEVICE)
        pah.load_state_dict(ckpt['pah'])
        pah.eval()
        normalizer = PAHNormalizer(3.0, 500.0 * 1.4142136, N_DRONES_TRAIN)
    return actor, pah, normalizer

@torch.no_grad()
def deterministic_action(actor, obs_np):
    """Deterministic policy: distribution MEAN, no sampling -- per
    docs/plans/02_experiment_protocol.md's eval procedure. NOTE: this
    differs from the stochastic (dist.sample()) actions used during this
    session's training-time eval logging -- see Cell 0."""
    obs_t = torch.tensor(obs_np, dtype=torch.float32, device=DEVICE)
    dist  = actor(obs_t)
    return dist.mean.cpu().numpy()

@torch.no_grad()
def alpha_for(pah, normalizer, tau, d, n):
    if pah is None:
        return np.full_like(tau, 0.5, dtype=np.float32)
    t  = torch.tensor(tau, dtype=torch.float32, device=DEVICE)
    dd = torch.tensor(d,   dtype=torch.float32, device=DEVICE)
    nn_ = torch.tensor(n,  dtype=torch.float32, device=DEVICE)
    x = normalizer.normalize(t, dd, nn_)
    alpha, _ = pah(x)
    return alpha.cpu().numpy()

def run_eval(actor, pah, normalizer, n_drones, n_obstacles, seeds, episodes_per_seed=20,
             world_size=500.0, max_steps=500):
    """Runs episodes_per_seed episodes for each seed, DETERMINISTIC policy,
    no gradient updates, no training. Returns success%, collision%, and
    the mean per-episode action std (a rough behavioral-variance proxy)."""
    successes, collisions, total = 0, 0, 0
    action_stds = []
    for seed in seeds:
        env = MultiUAVEnv(n_drones=n_drones, n_obstacles=n_obstacles, world_size=world_size,
                           max_speed=5.0, max_steps=max_steps, collision_radius=3.0,
                           target_radius=25.0, success_bonus=0.0, target_speed=1.0, seed=seed)
        for _ in range(episodes_per_seed):
            obs = env.reset()
            done = False
            ep_actions = []
            while not done:
                acts = deterministic_action(actor, obs)
                ep_actions.append(acts.copy())
                obs, rew, terminated, truncated, info = env.step(acts)
                done = terminated or truncated
            total += 1
            if info['all_targets_reached'] and not info['episode_collision']:
                successes += 1
            if info['episode_collision']:
                collisions += 1
            action_stds.append(np.std(np.concatenate(ep_actions, axis=0)))
    return dict(
        success=successes / total * 100, collision=collisions / total * 100,
        n_episodes=total, action_std=float(np.mean(action_stds)),
    )

m_actor, m_pah, m_norm = load_model(M_PATH, use_pah=True)
b4_actor, _, _         = load_model(B4_PATH, use_pah=False)

print('Loaded M (var-alpha) and B4 (fixed-alpha) seed-42 checkpoints. Inference only.')
print()
print('log_std (learned action noise, lower = more deterministic):')
print('  M  actor.log_std =', m_actor.log_std.data.numpy())
print('  B4 actor.log_std =', b4_actor.log_std.data.numpy())

# fresh, never-used-before seeds (not training seeds 42-46, not training-time eval seed 999)
FRESH_SEEDS = [2001, 2002, 2003, 2004, 2005]


In [ ]:
# Cell 6 -- Run (A) in-distribution and (B) out-of-distribution generalization evals

print('=' * 70)
print('(A) IN-DISTRIBUTION generalization: fresh unseen seeds, same config')
print('    (8 drones, 10 obstacles -- what both models were trained on)')
print('=' * 70)
res_m_a  = run_eval(m_actor, m_pah, m_norm, 8, 10, FRESH_SEEDS)
res_b4_a = run_eval(b4_actor, None, None,    8, 10, FRESH_SEEDS)
print('M  :', res_m_a)
print('B4 :', res_b4_a)
print()

print('=' * 70)
print('(B) OUT-OF-DISTRIBUTION: same fresh seeds, HARDER config')
print('    (8 drones, 16 obstacles -- 60% more than training, never seen)')
print('=' * 70)
res_m_b  = run_eval(m_actor, m_pah, m_norm, 8, 16, FRESH_SEEDS)
res_b4_b = run_eval(b4_actor, None, None,    8, 16, FRESH_SEEDS)
print('M  :', res_m_b)
print('B4 :', res_b4_b)


In [ ]:
# Cell 7 -- Summary comparison + save results

print('=' * 70)
print('SUMMARY')
print('=' * 70)
print('{:>28} | {:>10} | {:>10}'.format('Condition', 'M', 'B4'))
print('-' * 55)
print('{:>28} | {:>9.1f}% | {:>9.1f}%'.format('(A) in-dist success', res_m_a['success'], res_b4_a['success']))
print('{:>28} | {:>9.1f}% | {:>9.1f}%'.format('(A) in-dist collision', res_m_a['collision'], res_b4_a['collision']))
print('{:>28} | {:>9.1f}% | {:>9.1f}%'.format('(B) hard success', res_m_b['success'], res_b4_b['success']))
print('{:>28} | {:>9.1f}% | {:>9.1f}%'.format('(B) hard collision', res_m_b['collision'], res_b4_b['collision']))

drop_m  = res_m_a['success']  - res_m_b['success']
drop_b4 = res_b4_a['success'] - res_b4_b['success']
print()
print('Success drop (A -> B, in-dist -> hard):  M = {:.1f} pts | B4 = {:.1f} pts'.format(drop_m, drop_b4))
if drop_m > drop_b4 + 3:
    print('M degrades MORE than B4 under distribution shift -- consistent with the lower-entropy/less-exploration hypothesis.')
elif drop_b4 > drop_m + 3:
    print('B4 degrades MORE than M under distribution shift -- opposite of the hypothesis.')
else:
    print('No meaningful difference in degradation (within noise of a 100-episode sample) -- entropy '
          'difference does not show a clear effect on generalization here.')

import json
results = {
    'log_std': {'M': m_actor.log_std.data.numpy().tolist(), 'B4': b4_actor.log_std.data.numpy().tolist()},
    'in_distribution':     {'M': res_m_a, 'B4': res_b4_a},
    'out_of_distribution': {'M': res_m_b, 'B4': res_b4_b},
    'eval_seeds': FRESH_SEEDS,
    'note': 'Deterministic (mean-action) eval -- NOT the same methodology as this session\'s '
            'training-time stochastic eval logging. See Cell 0.',
}
with open('generalization_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved generalization_results.json')
